# Random Forest

In [3]:
from pathlib import Path

import gc
import os
import random
import pickle

import numpy as np
import pandas as pd
import optuna

from sklearn.metrics import roc_auc_score

In [19]:
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

PROJECT_DIR = Path("/Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton")
DATA_DIR = PROJECT_DIR / "data"
BASE_DIR = DATA_DIR

LOGS_DIR = PROJECT_DIR / "logs" / "random_forest_optuna_log"
MODELS_DIR = PROJECT_DIR / "models" / "random_forest_models"

LOGS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

required_files = [
    BASE_DIR / "y_train_sampled.parquet",
    BASE_DIR / "y_train_full.parquet",
    BASE_DIR / "y_val.parquet",
]

missing_files = [path for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "Не нашел подготовленные parquet-файлы:\n"
        + "\n".join(str(path) for path in missing_files)
    )

optuna.logging.set_verbosity(optuna.logging.WARNING)

print("PROJECT_DIR:", PROJECT_DIR)
print("BASE_DIR:", BASE_DIR)
print("LOGS_DIR:", LOGS_DIR)
print("MODELS_DIR:", MODELS_DIR)

PROJECT_DIR: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton
BASE_DIR: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/data
LOGS_DIR: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/logs/random_forest_optuna_log
MODELS_DIR: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/models/random_forest_models


In [5]:
# Наборы признаков
feature_sets = [
    "top300",
    "top500_clean",
    "top500_lgb_clean",
    "top500_magic_meta",
    "top500_micro_engineered",
]


def read_feature_matrix(folder, split):
    path = BASE_DIR / folder / f"X_{split}_{folder}.parquet"
    return pd.read_parquet(path)


def read_target(split):
    return pd.read_parquet(BASE_DIR / f"y_{split}.parquet").iloc[:, 0].values.ravel()

## Функция для оптимизации параметров с помощью optuna

In [ ]:
from sklearn.ensemble import RandomForestClassifier


def make_random_forest(trial):
    bootstrap = trial.suggest_categorical("bootstrap", [True, False])

    params = {
        "n_estimators": trial.suggest_int("n_estimatos", 200, 1000, step=100),
        "criterion": trial.suggest_categorical("criterion", ["gini", "entropy", "log_loss"]),
        "max_depth": trial.suggest_categorical("max_depth", [None, 8, 12, 16, 24, 32, 40]),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 80),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 40),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", 0.25, 0.5, None]),
        "class_weight": trial.suggest_categorical("class_weight", [None, "balanced", "balanced_subsample"]),
        "bootstrap": bootstrap,
        "n_jobs": -1,
        "random_state": SEED,
    }

    if bootstrap:
        params["max_samples"] = trial.suggest_float("max_samples", 0.45, 1.0)

    return RandomForestClassifier(**params)


def objective(trial, X_train, y_train, X_val, y_val):
    model = make_random_forest(trial)
    model.fit(X_train, y_train)

    preds = model.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val, preds)

## Настройка логирования

In [ ]:
def logging_callback(study, trial):
    set_name = study.user_attrs.get("set_name", "unknown")
    train_mode = study.user_attrs.get("train_mode", "unknown")
    log_file_path = LOGS_DIR / f"{set_name}_{train_mode}_optimization.log"

    with open(log_file_path, "a") as f:
        f.write(
            f"Trial {trial.number:03d} finished | "
            f"ROC-AUC: {trial.value:.5f} | "
            f"Best ROC-AUC so far: {study.best_value:.5f}\n"
        )

## Подбор параметров модели для каждого набора признаков. Для обучения используем усеченную обучающую выборку

In [8]:
# Загружаем общие таргеты
y_train_sampled = read_target("train_sampled")
y_val = read_target("val")

# Словари для сохранения результатов
best_params_sampled = {}
best_scores_sampled = {}

OPTUNA_N_TRIALS_SAMPLED = 30
OPTUNA_TIMEOUT_SAMPLED = 1800

In [9]:
# Главный цикл по наборам признаков
for folder in feature_sets:
    print(f"\n==================================================")
    print(f" НАЧАЛО ОПТИМИЗАЦИИ НАБОРА: {folder}")
    print(f"==================================================")

    # Подгружаем только нужные матрицы признаков для Optuna
    X_train_sampled = read_feature_matrix(folder, "train_sampled")
    X_val = read_feature_matrix(folder, "val")

    # Инициализируем исследование Optuna
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
    )

    # Сохраняем имя набора внутрь study, чтобы callback знал, куда писать логи
    study.set_user_attr("set_name", folder)
    study.set_user_attr("train_mode", "sampled")

    # Чистим старый лог-файл, если он остался от прошлых запусков
    log_file_path = LOGS_DIR / f"{folder}_sampled_optimization.log"
    if log_file_path.exists():
        log_file_path.unlink()

    # Запуск оптимизации
    study.optimize(
        lambda trial: objective(
            trial, X_train_sampled, y_train_sampled, X_val, y_val
        ),
        n_trials=OPTUNA_N_TRIALS_SAMPLED,
        timeout=OPTUNA_TIMEOUT_SAMPLED,
        callbacks=[logging_callback],
    )

    # Сохраняем лучшие результаты
    best_params_sampled[folder] = study.best_params
    best_scores_sampled[folder] = study.best_value

    # Выводим краткие итоги в консоль
    print(f"\n УСПЕШНО ЗАВЕРШЕНО ДЛЯ: {folder}")
    print(f" Лучший ROC-AUC на валидации: {study.best_value:.5f}")
    print(f" Подробный лог сохранен в: {log_file_path}")
    print(f" Лучшие параметры:")
    for param, value in study.best_params.items():
        print(f"   -> {param}: {value}")

    # Очистка памяти перед следующей итерацией
    del X_train_sampled, X_val, study
    gc.collect()

print("\n\n==================================================")
print(" ОПТИМИЗАЦИЯ ВСЕХ НАБОРОВ ЗАВЕРШЕНА!")
print("==================================================")


 НАЧАЛО ОПТИМИЗАЦИИ НАБОРА: top300

 УСПЕШНО ЗАВЕРШЕНО ДЛЯ: top300
 Лучший ROC-AUC на валидации: 0.64977
 Подробный лог сохранен в: /Users/pinta/hse/mlproject/random_forest_optuna_log/top300_sampled_optimization.log
 Лучшие параметры:
   -> bootstrap: False
   -> n_estimators: 900
   -> criterion: log_loss
   -> max_depth: 32
   -> min_samples_split: 26
   -> min_samples_leaf: 4
   -> max_features: sqrt
   -> class_weight: None

 НАЧАЛО ОПТИМИЗАЦИИ НАБОРА: top500_clean

 УСПЕШНО ЗАВЕРШЕНО ДЛЯ: top500_clean
 Лучший ROC-AUC на валидации: 0.65615
 Подробный лог сохранен в: /Users/pinta/hse/mlproject/random_forest_optuna_log/top500_clean_sampled_optimization.log
 Лучшие параметры:
   -> bootstrap: False
   -> n_estimators: 900
   -> criterion: log_loss
   -> max_depth: 16
   -> min_samples_split: 47
   -> min_samples_leaf: 7
   -> max_features: 0.5
   -> class_weight: None

 НАЧАЛО ОПТИМИЗАЦИИ НАБОРА: top500_lgb_clean

 УСПЕШНО ЗАВЕРШЕНО ДЛЯ: top500_lgb_clean
 Лучший ROC-AUC на валидации:

## Результаты оптимизации с обучением на усеченной обучающей выборке

In [10]:
# Выводим финальную сравнительную табличку в консоль
for folder in feature_sets:
    print(f"Dataset: {folder:<25} | Best ROC-AUC: {best_scores_sampled[folder]:.5f}")

Dataset: top300                    | Best ROC-AUC: 0.64977
Dataset: top500_clean              | Best ROC-AUC: 0.65615
Dataset: top500_lgb_clean          | Best ROC-AUC: 0.65729
Dataset: top500_magic_meta         | Best ROC-AUC: 0.65602
Dataset: top500_micro_engineered   | Best ROC-AUC: 0.65460


## Обучение моделей с лучшими параметрами на полной обучающей выборке и финальная проверка на валидации

In [11]:
# Общие таргеты
y_train_full = read_target("train_full")
y_val = read_target("val")

# Словари для хранения результатов финального тестирования
sampled_to_full_scores = {}

In [12]:
for folder, params in best_params_sampled.items():
    print(f"\n==================================================")
    print(f" ОБУЧЕНИЕ НА ПОЛНОМ TRAIN: {folder}")
    print(f"==================================================")

    X_train_full = read_feature_matrix(folder, "train_full")
    X_val = read_feature_matrix(folder, "val")

    final_params = params.copy()
    final_params["n_jobs"] = -1
    final_params["random_state"] = SEED

    # Если лучший вариант на sampled не использовал веса классов, на полном train добавляю балансировку.
    if final_params.get("class_weight") is None:
        final_params["class_weight"] = "balanced_subsample"

    final_model = RandomForestClassifier(**final_params)
    final_model.fit(X_train_full, y_train_full)

    # Проверяем итоговый скор
    val_preds = final_model.predict_proba(X_val)[:, 1]
    current_score = roc_auc_score(y_val, val_preds)
    sampled_to_full_scores[folder] = current_score

    print(f"Итоговый ROC-AUC после калибровки: {current_score:.5f}")

    del X_train_full, X_val, final_model
    gc.collect()


 ОБУЧЕНИЕ НА ПОЛНОМ TRAIN: top300
Итоговый ROC-AUC после калибровки: 0.61202

 ОБУЧЕНИЕ НА ПОЛНОМ TRAIN: top500_clean
Итоговый ROC-AUC после калибровки: 0.61541

 ОБУЧЕНИЕ НА ПОЛНОМ TRAIN: top500_lgb_clean
Итоговый ROC-AUC после калибровки: 0.61241

 ОБУЧЕНИЕ НА ПОЛНОМ TRAIN: top500_magic_meta
Итоговый ROC-AUC после калибровки: 0.61396

 ОБУЧЕНИЕ НА ПОЛНОМ TRAIN: top500_micro_engineered
Итоговый ROC-AUC после калибровки: 0.61322


## Результаты

In [13]:
for folder in best_params_sampled.keys():
    print(
        f"Dataset: {folder:<25} | New Final ROC-AUC: {sampled_to_full_scores[folder]:.5f}"
    )

Dataset: top300                    | New Final ROC-AUC: 0.61202
Dataset: top500_clean              | New Final ROC-AUC: 0.61541
Dataset: top500_lgb_clean          | New Final ROC-AUC: 0.61241
Dataset: top500_magic_meta         | New Final ROC-AUC: 0.61396
Dataset: top500_micro_engineered   | New Final ROC-AUC: 0.61322


## Подбор параметров модели для каждого набора признаков. Для обучения сразу используем полную обучающую выборку

In [14]:
# Загружаем общие таргеты
y_train_full = read_target("train_full")
y_val = read_target("val")

# Словари для сохранения результатов
best_params_per_set = {}
best_scores_per_set = {}

OPTUNA_N_TRIALS_FULL = 20
OPTUNA_TIMEOUT_FULL = 1800

In [15]:
# Главный цикл по наборам признаков
for folder in feature_sets:
    print(f"\n==================================================")
    print(f" НАЧАЛО ОПТИМИЗАЦИИ НАБОРА: {folder}")
    print(f"==================================================")

    # Подгружаем только нужные матрицы признаков для Optuna
    X_train_full = read_feature_matrix(folder, "train_full")
    X_val = read_feature_matrix(folder, "val")

    # Инициализируем исследование Optuna
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
    )

    # Сохраняем имя набора внутрь study, чтобы callback знал, куда писать логи
    study.set_user_attr("set_name", folder)
    study.set_user_attr("train_mode", "full")

    # Чистим старый лог-файл, если он остался от прошлых запусков
    log_file_path = LOGS_DIR / f"{folder}_full_optimization.log"
    if log_file_path.exists():
        log_file_path.unlink()

    # Запуск оптимизации
    study.optimize(
        lambda trial: objective(
            trial, X_train_full, y_train_full, X_val, y_val
        ),
        n_trials=OPTUNA_N_TRIALS_FULL,
        timeout=OPTUNA_TIMEOUT_FULL,
        callbacks=[logging_callback],
    )

    # Сохраняем лучшие результаты
    best_params_per_set[folder] = study.best_params
    best_scores_per_set[folder] = study.best_value

    # Выводим краткие итоги в консоль
    print(f"\n УСПЕШНО ЗАВЕРШЕНО ДЛЯ: {folder}")
    print(f" Лучший ROC-AUC на валидации: {study.best_value:.5f}")
    print(f" Подробный лог сохранен в: {log_file_path}")
    print(f" Лучшие параметры:")
    for param, value in study.best_params.items():
        print(f"   -> {param}: {value}")

    # Очистка памяти перед следующей итерацией
    del X_train_full, X_val, study
    gc.collect()

print("\n\n==================================================")
print(" ОПТИМИЗАЦИЯ ВСЕХ НАБОРОВ ЗАВЕРШЕНА!")
print("==================================================")


 НАЧАЛО ОПТИМИЗАЦИИ НАБОРА: top300

 УСПЕШНО ЗАВЕРШЕНО ДЛЯ: top300
 Лучший ROC-AUC на валидации: 0.64240
 Подробный лог сохранен в: /Users/pinta/hse/mlproject/random_forest_optuna_log/top300_full_optimization.log
 Лучшие параметры:
   -> bootstrap: False
   -> n_estimators: 900
   -> criterion: log_loss
   -> max_depth: 32
   -> min_samples_split: 26
   -> min_samples_leaf: 4
   -> max_features: sqrt
   -> class_weight: None

 НАЧАЛО ОПТИМИЗАЦИИ НАБОРА: top500_clean

 УСПЕШНО ЗАВЕРШЕНО ДЛЯ: top500_clean
 Лучший ROC-AUC на валидации: 0.65321
 Подробный лог сохранен в: /Users/pinta/hse/mlproject/random_forest_optuna_log/top500_clean_full_optimization.log
 Лучшие параметры:
   -> bootstrap: False
   -> n_estimators: 900
   -> criterion: log_loss
   -> max_depth: 32
   -> min_samples_split: 26
   -> min_samples_leaf: 4
   -> max_features: sqrt
   -> class_weight: None

 НАЧАЛО ОПТИМИЗАЦИИ НАБОРА: top500_lgb_clean

 УСПЕШНО ЗАВЕРШЕНО ДЛЯ: top500_lgb_clean
 Лучший ROC-AUC на валидации: 0.65

## Результаты оптимизации с обучением на полной обучающей выборке

In [16]:
# Выводим финальную сравнительную табличку в консоль
for folder in feature_sets:
    print(f"Dataset: {folder:<25} | Best ROC-AUC: {best_scores_per_set[folder]:.5f}")

Dataset: top300                    | Best ROC-AUC: 0.64240
Dataset: top500_clean              | Best ROC-AUC: 0.65321
Dataset: top500_lgb_clean          | Best ROC-AUC: 0.65297
Dataset: top500_magic_meta         | Best ROC-AUC: 0.65263
Dataset: top500_micro_engineered   | Best ROC-AUC: 0.64458


## Финальное обучение моделей с лучшими параметрами на полной обучающей выборке и их сохранение

In [17]:
# Загружаем общие таргеты
y_train_full = read_target("train_full")
y_val = read_target("val")

final_val_scores = {}

In [18]:
for folder, params in best_params_per_set.items():
    print(f"\n==================================================")
    print(f" ФИНАЛЬНОЕ ОБУЧЕНИЕ НА ПОЛНОМ ТРЕЙНЕ: {folder}")
    print(f"==================================================")

    # Загружаем полные матрицы признаков
    X_train_full = read_feature_matrix(folder, "train_full")
    X_val = read_feature_matrix(folder, "val")

    # Собираем параметры
    final_params = params.copy()
    final_params["n_jobs"] = -1
    final_params["random_state"] = SEED

    # Обучаем финальную модель
    final_model = RandomForestClassifier(**final_params)
    final_model.fit(X_train_full, y_train_full)

    # Проверяем итоговое качество на валидации
    val_preds = final_model.predict_proba(X_val)[:, 1]
    current_score = roc_auc_score(y_val, val_preds)
    final_val_scores[folder] = current_score

    print(f"Проверочный ROC-AUC на валидации: {current_score:.5f}")

    # Сохраняем модель
    model_filename = MODELS_DIR / f"rf_{folder}_{current_score:.5f}.pkl"
    with open(model_filename, "wb") as f:
        pickle.dump(final_model, f)
    print(f"Модель успешно сохранена: {model_filename}")

    # Очищаем оперативную память
    del X_train_full, X_val, final_model
    gc.collect()

print("\n\n==================================================")
print(" ОБУЧЕНИЕ И ИМПОРТ ВСЕХ МОДЕЛЕЙ ЗАВЕРШЕНЫ!")
print("==================================================")


 ФИНАЛЬНОЕ ОБУЧЕНИЕ НА ПОЛНОМ ТРЕЙНЕ: top300
Проверочный ROC-AUC на валидации: 0.64240
Модель успешно сохранена: /Users/pinta/hse/mlproject/models/random_forest_models/rf_top300_0.64240.pkl

 ФИНАЛЬНОЕ ОБУЧЕНИЕ НА ПОЛНОМ ТРЕЙНЕ: top500_clean
Проверочный ROC-AUC на валидации: 0.65321
Модель успешно сохранена: /Users/pinta/hse/mlproject/models/random_forest_models/rf_top500_clean_0.65321.pkl

 ФИНАЛЬНОЕ ОБУЧЕНИЕ НА ПОЛНОМ ТРЕЙНЕ: top500_lgb_clean
Проверочный ROC-AUC на валидации: 0.65297
Модель успешно сохранена: /Users/pinta/hse/mlproject/models/random_forest_models/rf_top500_lgb_clean_0.65297.pkl

 ФИНАЛЬНОЕ ОБУЧЕНИЕ НА ПОЛНОМ ТРЕЙНЕ: top500_magic_meta
Проверочный ROC-AUC на валидации: 0.65263
Модель успешно сохранена: /Users/pinta/hse/mlproject/models/random_forest_models/rf_top500_magic_meta_0.65263.pkl

 ФИНАЛЬНОЕ ОБУЧЕНИЕ НА ПОЛНОМ ТРЕЙНЕ: top500_micro_engineered
Проверочный ROC-AUC на валидации: 0.64458
Модель успешно сохранена: /Users/pinta/hse/mlproject/models/random_forest_mod